# Otsu Gaussian Weight Wild Horse Optimization (OTSU-GWWHO) for MRI Segmentation

This notebook implements the OTSU-GWWHO region-of-interest (ROI) segmentation for Alzheimer's Disease MRI scans. Otsu's within-class variance is the fitness function. The Gaussian Weight Wild Horse Optimization algorithm, inspired by the grazing, mating, dominance and leadership behaviour of wild horses, searches for the threshold that minimises it. A Gaussian weight on the horse–stallion distance controls each movement step. The optimal threshold produces a binary mask that separates brain tissue from the background, and the masked ROI is passed to the VGG16-DCNN classifier.

## Prerequisites

- Python 3.10+
- Required libraries: `numpy`, `opencv-python`, `pandas`, `matplotlib`, `tqdm`
- Outputs of `image_preprocessing.ipynb` and `data_augmentation.ipynb`

## Workflow

1. Compute the normalised histogram and the Otsu fitness (within-class variance)
2. Initialise the horse population with random thresholds and form groups with stallions
3. Update foal positions with Gaussian-weighted grazing and mating (crossover)
4. Update stallions with Gaussian-weighted leadership towards the water-hole (best threshold) and exchange leaders
5. Select the optimal threshold T* and generate the binary mask and ROI
6. Optionally plot convergence curves (fitness and threshold evolution)
7. Segment the augmented training set and the test set

In [ ]:
# ==========================================================
# IMPORTS & CONFIGURATION
# ==========================================================

import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

SPLIT_DIR = "./outputs/02_splits"
AUGMENTED_DIR = "./outputs/03_augmented"
SEGMENTED_DIR = "./outputs/04_segmented"
RANDOM_SEED = 42

POP_SIZE = 30        # N
MAX_ITER = 50        # T_max
PC = 0.13            # crossover (mating) probability
PS = 0.20            # stallion proportion
A_MIN = 0.01
A_MAX = 0.01
SIGMA_G = 32.0       # Gaussian parameter sigma_G
T_MIN, T_MAX = 0, 255
GRAY_LEVELS = 256

## Otsu Fitness Function

Computes the normalised histogram d(i) and the within-class variance σ²_w(t) = w_b(t)σ²_b(t) + w_f(t)σ²_f(t) for every threshold t. Lower values indicate a better separation of foreground and background. The values are precomputed once per image, so each fitness evaluation inside GWWHO is a lookup.

In [ ]:
# ==========================================================
# OTSU FITNESS (Within-Class Variance)
# ==========================================================

def normalized_histogram(img):
    """Eq. (3): d(i) = n_i / (R*C)"""
    hist = np.bincount(img.ravel(), minlength=GRAY_LEVELS).astype(np.float64)
    return hist / hist.sum()


def within_class_variance_table(d):
    """Eq. (4)-(6): sigma_w^2(t) = w_b*sigma_b^2 + w_f*sigma_f^2 for every threshold t."""
    i = np.arange(d.size, dtype=np.float64)
    w_b = np.cumsum(d)
    m_b = np.cumsum(i * d)
    s_b = np.cumsum(i * i * d)
    w_f = 1.0 - w_b
    m_f = m_b[-1] - m_b
    s_f = s_b[-1] - s_b
    with np.errstate(divide="ignore", invalid="ignore"):
        var_b = s_b / w_b - (m_b / w_b) ** 2
        var_f = s_f / w_f - (m_f / w_f) ** 2
        sigma_w = w_b * var_b + w_f * var_f
    sigma_w[(w_b <= 0) | (w_f <= 0)] = np.inf
    return np.nan_to_num(sigma_w, nan=np.inf)


def otsu_fitness(table, t):
    idx = np.clip(np.rint(t), 0, table.size - 1).astype(int)
    return table[idx]

## Gaussian Weight Wild Horse Optimization

The core GWWHO optimizer. Each horse is a candidate threshold. Foals graze around their stallion, with steps scaled by the Gaussian weight GW = exp(−d²/2σ²_G), or mate with a horse from another group with probability PC. Stallions move towards the water-hole (the best threshold) using Gaussian-weighted leadership. A foal replaces its stallion when it gives a lower within-class variance. Optionally generates convergence plots.

In [ ]:
# ==========================================================
# GWWHO THRESHOLD OPTIMIZATION WITH CONVERGENCE CURVE
# ==========================================================

def gaussian_weight(dist):
    """Eq. (12): GW = exp(-d^2 / (2*sigma_G^2))"""
    return np.exp(-(dist ** 2) / (2.0 * SIGMA_G ** 2))


def GWWHO_threshold(img, seed=None, plot_curve=False):
    rng = np.random.default_rng(seed)
    table = within_class_variance_table(normalized_histogram(img))
    fit = lambda x: otsu_fitness(table, x)
    clip = lambda x: np.clip(x, T_MIN, T_MAX)

    # Population initialisation, Eq. (8)
    pos = T_MIN + rng.random(POP_SIZE) * (T_MAX - T_MIN)
    cost = fit(pos)

    # Groups and stallions, Eq. (9): G = N x PS
    n_groups = max(1, int(round(POP_SIZE * PS)))
    order = np.argsort(cost)
    stallion, stallion_cost = pos[order[:n_groups]].copy(), cost[order[:n_groups]].copy()
    foals = pos[order[n_groups:]].copy()
    foal_group = np.arange(foals.size) % n_groups
    rng.shuffle(foal_group)
    foal_cost = fit(foals)

    # Initial water-hole (best threshold), Eq. (7)
    wp, wp_cost = stallion[np.argmin(stallion_cost)], stallion_cost.min()
    best_fit_history, best_T_history = [], []

    for t in range(MAX_ITER):
        # Adaptive search coefficient, Eq. (10)
        A_t = A_MIN + (1 - t / MAX_ITER) * (A_MAX - A_MIN)

        # Grazing / mating, Eq. (11)-(15)
        for i in range(foals.size):
            g = foal_group[i]
            if rng.random() > PC or n_groups == 1:
                d = abs(foals[i] - stallion[g])                                    # Eq. (11)
                GW = gaussian_weight(d)                                            # Eq. (12)
                R = rng.uniform(-2, 2)
                new = GW * (2 * A_t * np.cos(2 * np.pi * R * A_t)
                            * (stallion[g] - foals[i])) + stallion[g]              # Eq. (13)
            else:
                others = np.flatnonzero(foal_group != g)
                j = rng.choice(others)
                new = (foals[i] + foals[j]) / 2.0                                  # Eq. (14)
            foals[i] = clip(new)
            foal_cost[i] = fit(foals[i])

        # Gaussian-weighted leadership and leader exchange, Eq. (16)-(18)
        for g in range(n_groups):
            R = rng.uniform(-2, 2)
            GW = gaussian_weight(abs(wp - stallion[g]))
            step = 2 * A_t * np.cos(2 * np.pi * R * A_t) * GW * (wp - stallion[g])
            cand = clip(step + wp if rng.random() > 0.5 else step - wp)            # Eq. (16)
            c_cost = fit(cand)
            if c_cost < stallion_cost[g]:
                stallion[g], stallion_cost[g] = cand, c_cost
            members = np.flatnonzero(foal_group == g)
            if members.size:
                b = members[np.argmin(foal_cost[members])]
                if foal_cost[b] < stallion_cost[g]:                                # Eq. (18)
                    foals[b], stallion[g] = stallion[g], foals[b]
                    foal_cost[b], stallion_cost[g] = stallion_cost[g], foal_cost[b]

        # Update water-hole (global best), Eq. (17)
        g_best = np.argmin(stallion_cost)
        if stallion_cost[g_best] < wp_cost:
            wp, wp_cost = stallion[g_best], stallion_cost[g_best]
        best_fit_history.append(float(wp_cost))
        best_T_history.append(int(np.rint(wp)))

    # Final optimal threshold, Eq. (19)
    T_star = int(np.clip(np.rint(wp), T_MIN, T_MAX))

    if plot_curve:
        plt.figure(figsize=(9, 4))
        plt.subplot(1, 2, 1)
        plt.plot(best_fit_history, linewidth=3)
        plt.title("GWWHO Convergence (Fitness)")
        plt.xlabel("Iteration"); plt.ylabel("Within-class variance")
        plt.subplot(1, 2, 2)
        plt.plot(best_T_history, linewidth=3)
        plt.title("Threshold Evolution")
        plt.xlabel("Iteration"); plt.ylabel("Threshold")
        plt.tight_layout(); plt.show()

    return T_star, float(table[T_star])

## Segmentation Function

Applies the GWWHO-optimised threshold to an MRI image and returns the threshold, the binary mask and the segmented ROI.

In [ ]:
# ==========================================================
# SEGMENTATION FUNCTION
# ==========================================================

def GWWHO_segment(img, seed=None, plot_curve=False):
    """Eq. (20): I_s(r,c) = 1 if I(r,c) > T* else 0. Returns T*, mask and ROI."""
    if img.dtype != np.uint8:
        img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    if img.ndim == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    T_star, fitness = GWWHO_threshold(img, seed, plot_curve)
    mask = (img > T_star).astype(np.uint8)
    roi = img * mask
    return T_star, fitness, mask, roi

## Single Image Demo

Segments one pre-processed MRI slice and shows the input, mask, ROI and convergence curves.

In [ ]:
# ==========================================================
# SINGLE IMAGE DEMO
# ==========================================================

demo_path = pd.read_csv(os.path.join(SPLIT_DIR, "test.csv"))["path"].iloc[0]
img = cv2.imread(demo_path, cv2.IMREAD_GRAYSCALE)
T_star, fitness, mask, roi = GWWHO_segment(img, seed=RANDOM_SEED, plot_curve=True)

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a, im, title in zip(ax, [img, mask, roi], ["Input MRI", f"Mask (T* = {T_star})", "Segmented ROI"]):
    a.imshow(im, cmap="gray"); a.set_title(title); a.axis("off")
plt.tight_layout(); plt.show()

## Batch Segmentation

Segments the augmented training set and the untouched test set, and saves the ROI images, masks and thresholds.

In [ ]:
# ==========================================================
# BATCH SEGMENTATION
# ==========================================================

def segment_csv(in_csv, out_csv, out_dir):
    df = pd.read_csv(in_csv)
    rows = []
    for idx, r in tqdm(df.iterrows(), total=len(df), desc=os.path.basename(in_csv)):
        img = cv2.imread(r["path"], cv2.IMREAD_GRAYSCALE)
        T_star, fitness, mask, roi = GWWHO_segment(img, seed=RANDOM_SEED + idx)

        stem = os.path.splitext(os.path.basename(r["path"]))[0]
        roi_dir = os.path.join(out_dir, "roi", r["label"])
        mask_dir = os.path.join(out_dir, "mask", r["label"])
        os.makedirs(roi_dir, exist_ok=True)
        os.makedirs(mask_dir, exist_ok=True)
        roi_path = os.path.join(roi_dir, stem + ".png")
        mask_path = os.path.join(mask_dir, stem + ".png")
        cv2.imwrite(roi_path, roi)
        cv2.imwrite(mask_path, mask * 255)
        rows.append({**r.to_dict(), "roi_path": roi_path, "mask_path": mask_path,
                     "threshold": T_star, "otsu_fitness": fitness})
    pd.DataFrame(rows).to_csv(out_csv, index=False)


os.makedirs(SEGMENTED_DIR, exist_ok=True)
segment_csv(os.path.join(AUGMENTED_DIR, "train_augmented.csv"),
            os.path.join(SEGMENTED_DIR, "train_segmented.csv"), os.path.join(SEGMENTED_DIR, "train"))
segment_csv(os.path.join(SPLIT_DIR, "test.csv"),
            os.path.join(SEGMENTED_DIR, "test_segmented.csv"), os.path.join(SEGMENTED_DIR, "test"))